# Trusted sealed-corpus analysis template

The external runner must verify this output-free template's preregistered SHA-256 before execution. The template then opens each trusted bundle file exactly once, authenticates those exact bytes against the external template, driver, and preregistration SHA-256 receipts, and compiles/executes only the verified driver bytes. Before admission it also authenticates the sealed attempt-ledger and raw-manifest byte digests plus the expected M/tree/C/fingerprint/runtime-control/capture/schedule/protocol bindings supplied by the external runner. The terminal notebook receipt binds every digest, the ordered admitted report hashes, and admission traceability to the analysis schema, explicit evidence/action status, canonical output paths, and p95 impossibility receipt. Run in the network-disabled GJC RLM sandbox with the corpus on an externally enforced immutable read-only mount and a separate bounded writable output directory.

In [ ]:
import os
import sys

ENVIRONMENT_PARAMETERS = {
    name: os.environ[name]
    for name in (
        "GJC_PERF_CORPUS_BUNDLE_DIR",
        "GJC_PERF_CORPUS_INPUT_DIR",
        "GJC_PERF_CORPUS_OUTPUT_DIR",
        "GJC_PERF_CORPUS_EXPECTED_GIT_SHA",
        "GJC_PERF_CORPUS_EXPECTED_TREE_SHA",
        "GJC_PERF_CORPUS_EXPECTED_CLOSURE_DIGEST",
        "GJC_PERF_CORPUS_EXPECTED_WORKTREE_FINGERPRINT",
        "GJC_PERF_CORPUS_EXPECTED_RUNTIME_CONTROL_IDENTITY",
        "GJC_PERF_CORPUS_EXPECTED_CAPTURE_ID",
        "GJC_PERF_CORPUS_EXPECTED_SCHEDULE_DIGEST",
        "GJC_PERF_CORPUS_EXPECTED_PROTOCOL_DIGEST",
        "GJC_PERF_CORPUS_TEMPLATE_SHA256",
        "GJC_PERF_CORPUS_DRIVER_SHA256",
        "GJC_PERF_CORPUS_PREREGISTRATION_SHA256",
        "GJC_PERF_CORPUS_ATTEMPT_LEDGER_SHA256",
        "GJC_PERF_CORPUS_RAW_MANIFEST_SHA256",
        "GJC_PERF_CORPUS_INPUT_MOUNT_READ_ONLY",
    )
}

def resolve_search_path(search_entry):
    return os.path.realpath(search_entry if search_entry else os.getcwd())

def is_at_or_below(candidate, root):
    try:
        return os.path.commonpath((candidate, root)) == root
    except ValueError:
        return False

untrusted_import_roots = tuple(
    os.path.realpath(ENVIRONMENT_PARAMETERS[name])
    for name in ("GJC_PERF_CORPUS_BUNDLE_DIR", "GJC_PERF_CORPUS_INPUT_DIR")
)
for search_entry in sys.path:
    resolved_search_entry = resolve_search_path(search_entry)
    if any(is_at_or_below(resolved_search_entry, root) for root in untrusted_import_roots):
        raise RuntimeError("bundle and input directories and their descendants must not be on Python import search paths")

import hashlib
import stat
from pathlib import Path
bundle_dir = Path(ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_BUNDLE_DIR"])
input_dir = Path(ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_INPUT_DIR"])
output_dir = Path(ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_OUTPUT_DIR"])
driver_path = bundle_dir / "perf-corpus-rlm-analysis.py"
preregistration_path = bundle_dir / "perf-corpus-preregistration.json"

def require_sha256(value, label):
    normalized = value.lower()
    if len(normalized) != 64 or any(character not in "0123456789abcdef" for character in normalized):
        raise RuntimeError(f"invalid external SHA-256 receipt: {label}")
    return normalized

def read_verified_once(path, expected, maximum_bytes):
    expected = require_sha256(expected, path.name)
    path_info = path.lstat()
    if not stat.S_ISREG(path_info.st_mode) or stat.S_ISLNK(path_info.st_mode):
        raise RuntimeError(f"untrusted bundle path: {path}")
    flags = os.O_RDONLY
    if hasattr(os, "O_NOFOLLOW"):
        flags |= os.O_NOFOLLOW
    descriptor = os.open(path, flags)
    try:
        before = os.fstat(descriptor)
        raw = bytearray()
        while len(raw) <= maximum_bytes:
            chunk = os.read(descriptor, min(1024 * 1024, maximum_bytes + 1 - len(raw)))
            if not chunk:
                break
            raw.extend(chunk)
        after = os.fstat(descriptor)
    finally:
        os.close(descriptor)
    if (
        not stat.S_ISREG(before.st_mode)
        or len(raw) > maximum_bytes
        or (before.st_dev, before.st_ino, before.st_size, before.st_mtime_ns)
        != (after.st_dev, after.st_ino, after.st_size, after.st_mtime_ns)
        or len(raw) != before.st_size
    ):
        raise RuntimeError(f"trusted bundle file is unsafe or changed while reading: {path.name}")
    exact_bytes = bytes(raw)
    if hashlib.sha256(exact_bytes).hexdigest() != expected:
        raise RuntimeError(f"SHA-256 mismatch: {path.name}")
    return exact_bytes

template_sha256 = require_sha256(
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_TEMPLATE_SHA256"],
    "perf-corpus-rlm-template.ipynb",
)
if ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_INPUT_MOUNT_READ_ONLY"] != "1":
    raise RuntimeError("external runner must attest an immutable read-only input mount")
bundle_info = bundle_dir.lstat()
input_info = input_dir.lstat()
if not stat.S_ISDIR(bundle_info.st_mode) or stat.S_ISLNK(bundle_info.st_mode):
    raise RuntimeError("bundle must be a real directory")
if not stat.S_ISDIR(input_info.st_mode) or stat.S_ISLNK(input_info.st_mode):
    raise RuntimeError("input must be a real directory")
if input_info.st_mode & 0o222:
    raise RuntimeError("input directory must have no write permission bits in addition to the read-only mount")
if input_dir.resolve() == output_dir.resolve():
    raise RuntimeError("input and output directories must differ")

driver_bytes = read_verified_once(
    driver_path,
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_DRIVER_SHA256"],
    1024 * 1024,
)
preregistration_bytes = read_verified_once(
    preregistration_path,
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_PREREGISTRATION_SHA256"],
    1024 * 1024,
)
driver_namespace = {
    "__builtins__": __builtins__,
    "__file__": f"<verified-driver-sha256:{ENVIRONMENT_PARAMETERS['GJC_PERF_CORPUS_DRIVER_SHA256'].lower()}>",
    "__name__": "gjc_trusted_perf_corpus_analysis",
}
driver_code = compile(
    driver_bytes,
    driver_namespace["__file__"],
    "exec",
    dont_inherit=True,
)
exec(driver_code, driver_namespace)
completed = driver_namespace["run_analysis"](
    input_dir,
    output_dir,
    preregistration_bytes,
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_EXPECTED_GIT_SHA"],
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_EXPECTED_TREE_SHA"],
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_EXPECTED_CLOSURE_DIGEST"],
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_EXPECTED_WORKTREE_FINGERPRINT"],
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_EXPECTED_RUNTIME_CONTROL_IDENTITY"],
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_EXPECTED_CAPTURE_ID"],
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_EXPECTED_SCHEDULE_DIGEST"],
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_EXPECTED_PROTOCOL_DIGEST"],
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_DRIVER_SHA256"],
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_PREREGISTRATION_SHA256"],
    template_sha256,
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_ATTEMPT_LEDGER_SHA256"],
    ENVIRONMENT_PARAMETERS["GJC_PERF_CORPUS_RAW_MANIFEST_SHA256"],
)
display({
    "analysisSchema": completed["result"]["schema"],
    "evidenceStatus": completed["result"]["evidenceStatus"],
    "actionDecision": completed["result"]["actionDecision"],
    "hashBindings": completed["result"]["hashBindings"],
    "admissionTraceability": completed["result"]["admissionTraceability"],
    "p95MethodReceipt": completed["result"]["claimPolicy"]["p95"],
    "resultJsonPath": completed["resultJsonPath"],
    "resultMarkdownPath": completed["resultMarkdownPath"],
})
